# P2 acting-model activation screen — A100 / bf16

**Protocol signature: `P2_EXACT_TOKEN_V3_COMPLETION_GATE`.** This notebook processes only
`ai_liar` and `sandbagging`. If output mentions `roleplaying`,
`insider_trading`, or `LABEL_MODE=self_judge`, interrupt it: that is the
obsolete launcher and its result is invalid.

This notebook runs path (b): regenerate responses with one acting model and
capture that same model's hidden states on the **exact generated token IDs**.

The default run is deliberately a two-task screen over `ai_liar` and
`sandbagging`. Apollo supplies balanced intended honest/deceptive conditions
for those tasks. Those conditions are not proof that this 8B model actually
lied or sandbagged, so the first run is **not eligible for a formal P2
supported/refuted verdict**. Independently grade the saved generations and
rerun analysis with the label file before calling it realized deception.
A full P2 verdict also needs four task families.

What this run settles now:

- acting model equals probed model;
- no decode/re-tokenize activation mismatch;
- generated-token activations exclude prompt tokens;
- per-response checkpoints survive Colab disconnects;
- prototype and logistic transfer matrices use the same samples;
- prompt-only leakage and null controls are reported;
- the internal-minus-fixed-behavioral delta receives a paired bootstrap CI.

Set **Runtime → Change runtime type → A100 GPU**, then run cells in order.
High-RAM host memory is optional; the runner streams the model directly to
the GPU and processes one response at a time. T4, L4, quantized, and fp16
runs are infrastructure smokes only.

In [ ]:
# Install the versions used to validate the local runner. Colab supplies torch/CUDA.
print("Protocol: P2_EXACT_TOKEN_V3_COMPLETION_GATE")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"


In [ ]:
# Mount Drive and load the versioned runner staged beside this notebook.
from google.colab import drive
drive.mount("/content/drive")
import shutil
STAGED_RUNNER = "/content/drive/MyDrive/phi-map/p2-v3-launch/p2_acting_model_pilot.py"
assert __import__("os").path.exists(STAGED_RUNNER), f"Missing {STAGED_RUNNER}"
shutil.copy2(STAGED_RUNNER, "/content/p2_acting_model_pilot.py")
print("Runner staged: OK")


In [ ]:
# Persist checkpoints to Drive so a Colab disconnect does not destroy the run.
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/p2-llama31-8b-seed17-v2"
CAP_PER_TASK = 60
MAX_NEW_TOKENS = 1024
SEED = 17
BOOTSTRAP = 5000

# Put HF_TOKEN in Colab's Secrets panel; do not paste it into the notebook.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"


In [ ]:
# Hardware gate. The runner repeats this check and rejects non-bf16/non-A100 runs.
# First verify the secret can read the gated model without printing the token.
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()


In [ ]:
# Generate, checkpoint exact token IDs, capture activations, and analyze.
# Re-running resumes completed responses instead of starting over.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
cmd = [
    sys.executable, "/content/p2_acting_model_pilot.py",
    "--model", ACTING_MODEL,
    "--work-dir", WORK_DIR,
    "--cap-per-task", str(CAP_PER_TASK),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--seed", str(SEED),
    "--bootstrap", str(BOOTSTRAP),
]
subprocess.run(cmd, check=True, env=env)


In [ ]:
# Compact result view. The full matrices and controls stay in Drive.
import json
result_path = f"{WORK_DIR}/p2_pilot_result.json"
result = json.load(open(result_path))
print(json.dumps({
    "status": result["status"],
    "formal_p2_decision": result["formal_p2_decision"],
    "ineligibility": result["formal_ineligibility_reasons"],
    "primary": result["primary"],
    "generation_quality": result["generation_quality"],
    "prompt_only_cross_task": result["prompt_only_cross_task_leakage_control"],
    "controls": result["controls"],
}, indent=2))
print("full artifact:", result_path)


## After independent outcome grading

Create `p2_outcome_labels.json` with this shape:

```json
{
  "label_source": "fixed external grader and version",
  "labels": {
    "ai_liar:0000": "deceptive",
    "sandbagging:0004": "honest"
  }
}
```

Upload it, then rerun without loading the 8B model:

```python
from google.colab import files
files.upload()
subprocess.run(cmd + [
    "--analysis-only",
    "--label-file", "/content/p2_outcome_labels.json",
], check=True, env=env)
```

Rows absent from the independent label file are excluded; intended-condition
labels are never silently mixed into the outcome-labeled rerun.